In [2]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_sensor_messages_timestamped_stage,
    validate_sensor_messages_timestamped_stage,
)


In [3]:
SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IF_EXISTS_FLAG = str("replace")
CHUNK_SIZE = int(250000)

SIMULATION_TIME_CONFIG_TABLE_NAME = str("simulation_timing_config")
SIMULATION_START_DATETIME = str("2026-03-19 08:00:00+00:00")
SIMULATION_SAMPLING_INTERVAL_SECONDS = float(1.0)

TIMESTAMPED_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_stage")
TIMESTAMPED_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_timestamped_stage")

In [4]:

engine = get_engine_from_env()


In [5]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


'simulation_timing_config'

In [6]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

Timing config ready.


----

In [7]:
timestamped_table_name = build_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


[chunk] 1 | source rows 0 to 249,999
[timestamp] wrote chunk 1 with 250,000 rows
[chunk] 2 | source rows 250,000 to 499,999
[timestamp] wrote chunk 2 with 250,000 rows
[chunk] 3 | source rows 500,000 to 749,999
[timestamp] wrote chunk 3 with 250,000 rows
[chunk] 4 | source rows 750,000 to 999,999
[timestamp] wrote chunk 4 with 250,000 rows
[chunk] 5 | source rows 1,000,000 to 1,249,999
[timestamp] wrote chunk 5 with 250,000 rows
[chunk] 6 | source rows 1,250,000 to 1,499,999
[timestamp] wrote chunk 6 with 250,000 rows
[chunk] 7 | source rows 1,500,000 to 1,749,999
[timestamp] wrote chunk 7 with 250,000 rows
[chunk] 8 | source rows 1,750,000 to 1,999,999
[timestamp] wrote chunk 8 with 250,000 rows
[chunk] 9 | source rows 2,000,000 to 2,249,999
[timestamp] wrote chunk 9 with 250,000 rows
[chunk] 10 | source rows 2,250,000 to 2,499,999
[timestamp] wrote chunk 10 with 250,000 rows
[chunk] 11 | source rows 2,500,000 to 2,749,999
[timestamp] wrote chunk 11 with 250,000 rows
[chunk] 12 | sour

----

In [8]:
validation_dataframe = validate_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)



In [9]:
display(validation_dataframe)

,row_count,distinct_observation_count,min_observation_timestamp,max_observation_timestamp,distinct_observation_timestamp_count
0,39000000,750000,2026-03-19 08:00:00+00:00,2026-03-28 00:19:59+00:00,750000


----

In [10]:

sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(sample_dataframe)

,observation_index,observation_timestamp,message_sequence_index,sensor_name,sensor_index,sensor_value,stream_state,phase,meta_episode_id
0,1,2026-03-19 08:00:00+00:00,0,sensor_41,41,36.767719,normal,normal,0
1,1,2026-03-19 08:00:00+00:00,1,sensor_49,49,53.702847,normal,normal,0
2,1,2026-03-19 08:00:00+00:00,2,sensor_46,46,43.532345,normal,normal,0
3,1,2026-03-19 08:00:00+00:00,3,sensor_32,32,928.790527,normal,normal,0
4,1,2026-03-19 08:00:00+00:00,4,sensor_18,18,2.743295,normal,normal,0
...,...,...,...,...,...,...,...,...,...
99,2,2026-03-19 08:00:01+00:00,47,sensor_13,13,4.297912,normal,normal,0
100,2,2026-03-19 08:00:01+00:00,48,sensor_11,11,43.319096,normal,normal,0
101,2,2026-03-19 08:00:01+00:00,49,sensor_07,7,15.981363,normal,normal,0
102,2,2026-03-19 08:00:01+00:00,50,sensor_08,8,15.264808,normal,normal,0


In [11]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_timestamps_in_observation,
        MIN(observation_timestamp) AS observation_timestamp_min,
        MAX(observation_timestamp) AS observation_timestamp_max
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)


In [12]:

display(timestamp_check_dataframe)

,observation_index,message_count,distinct_timestamps_in_observation,observation_timestamp_min,observation_timestamp_max
0,1,52,1,2026-03-19 08:00:00+00:00,2026-03-19 08:00:00+00:00
1,2,52,1,2026-03-19 08:00:01+00:00,2026-03-19 08:00:01+00:00
2,3,52,1,2026-03-19 08:00:02+00:00,2026-03-19 08:00:02+00:00
3,4,52,1,2026-03-19 08:00:03+00:00,2026-03-19 08:00:03+00:00
4,5,52,1,2026-03-19 08:00:04+00:00,2026-03-19 08:00:04+00:00
5,6,52,1,2026-03-19 08:00:05+00:00,2026-03-19 08:00:05+00:00
6,7,52,1,2026-03-19 08:00:06+00:00,2026-03-19 08:00:06+00:00
7,8,52,1,2026-03-19 08:00:07+00:00,2026-03-19 08:00:07+00:00
8,9,52,1,2026-03-19 08:00:08+00:00,2026-03-19 08:00:08+00:00
9,10,52,1,2026-03-19 08:00:09+00:00,2026-03-19 08:00:09+00:00
